# 02b · Toro plano: protocolo de cierre

Mismo protocolo que el cierre del Swiss roll (notebook 05): encoder DMAP congelado, `vae` con kl = 10⁻³ y 10 000 pasos de Adam, entrenado **desde cero** y por **continuación en m** (orden espectral), réplicas independientes (datos y partición cambian con la semilla), recuperación de la carta medida con kNN (latente → carta) frente a su piso, tasas con intervalo de Clopper-Pearson al 95 %.

**Geometría.** T² plano en R⁴, x = (cos θ₁, sin θ₁, 0.6 cos θ₂, 0.6 sin θ₂). Las frecuencias del Laplaciano son k²/R²: 1, 1 (θ₁), 2.78, 2.78 (θ₂), 4, 4 (2θ₁)… Los pares (cos, sin) son exactamente degenerados, así que el segundo ángulo entra en las **posiciones 3–4**. Carta = (cos θ₁, sin θ₁, cos θ₂, sin θ₂).

Hipótesis y criterios, **fijados antes de correr**:

| | Pregunta | Diseño | Criterio |
|---|---|---|---|
| **T1** | ¿El prefijo espectral contiene la carta donde predice el teorema? | oráculo afín, 16 réplicas | chart MSE_z mediana: m=2 → **0.25** (falta θ₂), m=3 → **0.125** (medio par), ambos ±10 %; m≥4 → < 0.05 (el piso cae ≥ 5×). Ambiente (lectura afín de x): 0.09 y 0.045 |
| **T2** | ¿La continuación espectral despliega la carta de forma fiable? | latente 4 (T² se sumerge en R⁴), 16 réplicas | `vae_warm` desplegado, límite inferior Clopper-Pearson > 0.6 para todo m ≥ 4 |
| **T3** | Control topológico | latente 2: T² **no** se sumerge en R² | ninguna m con despliegue fiable (límite superior < 0.5). Si latente 2 "despliega", el umbral kNN es demasiado laxo para el toro |
| **T4** | ¿El mecanismo es el orden espectral? | igual que T2 con los modos en orden aleatorio (8 réplicas) | en m = M = 64 (mismos modos, misma clase): Fisher exacto unilateral p < 0.05 para `vae_warm`; el control `vae` desde cero no debe diferir |
| **T5** | Comparador convencional | `plainvae` (MLP sobre x), latentes 4 y 2 | se reporta la misma métrica |

β se elige **sin etiquetas** con el defecto de degeneración de los pares (cos, sin) del espectro; el criterio de atajos del Swiss roll no aplica en el toro (geodésica/cuerda ≤ π/2 < 2). Mismo n en el barrido y en el experimento.

Duración estimada ~5.5 h en dos T4 (la celda del plan imprime el detalle).

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'protocol.py', 'config_beta_sweep.json',
        'config_beta_sweep_torus.json', 'config_beta_sweep_mnist.json', 'config_beta_sweep_alanine.json']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, (f'Faltan archivos del paquete: {missing_code}. Este notebook necesita el '
                          'paquete v5 (dmap_vae_kaggle_v5.zip): sustituye el Dataset anterior.')

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## Controles del paquete (una GPU, ~3 min)

Compila FlashDiffusion y ejecuta `checks.py`: kernel contra matriz densa, convención de signo, métrica kNN, continuación con ceros, paso AE→VAE y los controles nuevos de v5 (regla de degeneración del toro, regla argmin, sonda de etiquetas de MNIST, rotación de trayectorias de alanina). Si falla, no sigas.

In [ ]:
import time
t0 = time.time()
r = subprocess.run([sys.executable, str(ROOT / 'checks.py'), '--backend', 'flash', '--device', 'cuda:0',
                    '--out', str(ROOT / 'checks_02b.json')],
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:]); print(r.stderr[-6000:])
    raise RuntimeError('No continuar: falló un control del paquete.')
checks = json.loads((ROOT / 'checks_02b.json').read_text())
print({k: v for k, v in checks.items() if isinstance(v, str)})
print(f"signo: {checks['sign_overlap_min']:.6f} | kNN suave/plegada: {checks['knn_smooth_ratio']:.2f} / "
      f"{checks['knn_folded_ratio']:.0f} veces el piso | {(time.time()-t0)/60:.1f} min")

## β sin etiquetas (mismo n que el experimento, ~10–15 min)

Regla: el factor que minimiza el peor caso (sobre semillas de datos) del **defecto de degeneración** = separación máxima dentro de los tres primeros pares / salto mínimo entre pares, entre los factores que pasan los controles espectrales en todas las semillas. La banda (≤ 2× el mínimo) se reporta como equivalente. Las columnas `oracle_*` usan la carta y son solo diagnóstico.

In [ ]:
import time
import pandas as pd
from IPython.display import display, Image
import protocol as P

N = 3000                      # el MISMO n en el barrido y en el experimento: beta depende de la densidad
cfg_sweep = json.loads((ROOT / 'config_beta_sweep_torus.json').read_text())
cfg_sweep.update(n=N)
sel = P.run_sweep(cfg_sweep, ROOT, 'beta_sweep_torus')
FACTOR = sel['selected_factor']
assert FACTOR is not None, 'Ningún factor pasó los controles espectrales en todas las semillas: amplía la lista.'
if 'warning' in sel:
    print('ATENCIÓN:', sel['warning'])
print(f"factor seleccionado = {FACTOR} | banda equivalente = {sel['equivalent_band']}")

SWEEP = Path(cfg_sweep['out'])
sw = pd.read_csv(SWEEP / 'beta_sweep.csv')
cols = [c for c in ['kernel_length', 'degeneracy_defect', 'graph_shortcut_mass', 'perron_gap', 'min_eigenvalue',
                    'nystrom_training_relative_error', 'oracle_val_m2', 'oracle_val_m3', 'oracle_val_m4',
                    'ambient_val_m2', 'ambient_val_m4'] if c in sw]
display(sw.groupby('factor')[cols].agg(lambda s: s.max() if s.name == 'degeneracy_defect' else s.mean())
        .style.format('{:.3g}').set_caption('defecto: peor semilla; resto: media. oracle_*: diagnóstico con la carta'))
print('Predicción (oráculo afín, carta): m=2 -> 0.25, m=3 -> 0.125, m>=4 -> < 0.05 (error de muestra finita)')
display(Image(filename=str(SWEEP / 'beta_sweep.png')))

## Plan y estimación de tiempo

El orden de `PLAN` es la prioridad. Ajusta `SECONDS_PER_MODEL` con el tiempo real de la primera configuración (en el Swiss roll fueron ~80 s por modelo de 10 000 pasos).

In [ ]:
import math
import protocol as P
BASE = json.loads((ROOT / 'config_torus.json').read_text())
BASE.update(n=N, alpha=1.0)
COMMON = dict(P.COMMON)
LADDER = [2, 3, 4, 8, 16, 32, 64]          # 3 parte el par de theta_2: comprobación intermedia del teorema
M = max(LADDER)
PLAN = [
    dict(tag='A_k4_spectral', latent=4, beta_factor=FACTOR, seeds=list(range(16)), modes=LADDER,
         models=['vae'], m_protocols=['scratch', 'warm'], mode_order='spectral', chart_decoder_epochs=10_000),
    dict(tag='C_k2_spectral', latent=2, beta_factor=FACTOR, seeds=list(range(8)), modes=LADDER,
         models=['vae'], m_protocols=['scratch', 'warm'], mode_order='spectral'),
    dict(tag='B_k4_shuffled', latent=4, beta_factor=FACTOR, seeds=list(range(8)), modes=LADDER,
         models=['vae'], m_protocols=['scratch', 'warm'], mode_order='shuffled', mode_order_seed=12345),
    dict(tag='D_plainvae_k4', latent=4, beta_factor=FACTOR, seeds=list(range(8)), modes=[LADDER[0]],
         models=['plainvae'], m_protocols=['scratch'], oracle_decoder=False),
    dict(tag='D_plainvae_k2', latent=2, beta_factor=FACTOR, seeds=list(range(8)), modes=[LADDER[0]],
         models=['plainvae'], m_protocols=['scratch'], oracle_decoder=False),
]
SECONDS_PER_MODEL = 80
print('pasos de Adam por modelo =', max(COMMON['epochs'] * math.ceil(0.6 * N / COMMON['batch']), COMMON['min_steps']))
P.show_plan(PLAN, 'torus', SECONDS_PER_MODEL, COMMON)

## Ejecutar

Se puede re-ejecutar: las configuraciones completas se omiten y las parciales se señalan (bórralas o cambia el `tag`). Para una corrida larga conviene **Save Version → Save & Run All (Commit)**.

In [ ]:
import protocol as P
hechas, problemas = P.run_plan(PLAN, BASE, ROOT, common=COMMON)

## Análisis

"Desplegada" = kNN(latente → carta) ≤ 10 × el piso del mismo estimador sobre la carta verdadera. Las tablas incluyen la fila de inicio de cada cadena `warm` (= el modelo desde cero en m = 2), así la curva empieza donde empieza el camino. Tasas con Clopper-Pearson al 95 %.

In [ ]:
import ast
import numpy as np
import pandas as pd
from IPython.display import display, Image
import matplotlib.pyplot as plt
import protocol as P
R = {p['tag']: P.load(p['out']) for p in PLAN}
fmt = {'tasa': '{:.0%}', 'ic95_inf': '{:.2f}', 'ic95_sup': '{:.2f}', 'mse_x_mediana': '{:.4g}',
       'knn_ratio_mediana': '{:.3g}'}
A, B, C = R['A_k4_spectral'], R['B_k4_shuffled'], R['C_k2_spectral']

# ---- T1: información del prefijo frente a la predicción ----
PRED_Z = {2: .25, 3: .125}
PRED_X = {2: .09, 3: .045}
if A is not None:
    info = P.information_table(A).join(
        P.information_table(A, 'mse_x', 'affine_x').add_prefix('ambiente_'))
    display(info.style.format('{:.3g}').set_caption('T1 · oráculo afín (carta) y lectura afín de x (ambiente), cuartiles'))
    med = info['mediana']
    ok1 = all(abs(med.loc[m]/v - 1) < .10 for m, v in PRED_Z.items() if m in med) and \
          all(med.loc[m] < .05 for m in med.index if m >= 4)
    for m, v in PRED_Z.items():
        if m in med:
            print(f'  m={m}: carta {med.loc[m]:.4f} (teorema {v}) | ambiente {info.ambiente_mediana.loc[m]:.4f} (teorema {PRED_X[m]})')
    print(f"Criterio T1: {'CUMPLIDO' if ok1 else 'NO cumplido'}")

# ---- T2 y T3: despliegue con latente 4 y latente 2 ----
tablas = {}
for tag, lab in (('A_k4_spectral', 'A latente 4'), ('C_k2_spectral', 'C latente 2'), ('B_k4_shuffled', 'B latente 4, aleatorio'),
                 ('D_plainvae_k4', 'plainvae latente 4'), ('D_plainvae_k2', 'plainvae latente 2')):
    if R[tag] is not None:
        tablas[tag] = P.unfold_table(R[tag], lab)
        display(tablas[tag].style.format(fmt).set_caption(lab))
if 'A_k4_spectral' in tablas:
    t = tablas['A_k4_spectral']
    w = t[(t.model == 'vae_warm') & (t.m >= 4)]
    print(f"Criterio T2 (IC inf > 0.6 para vae_warm, m >= 4): {'CUMPLIDO' if len(w) and (w.ic95_inf > .6).all() else 'NO cumplido'}")
if 'C_k2_spectral' in tablas:
    t = tablas['C_k2_spectral']
    print(f"Criterio T3 (latente 2 sin despliegue fiable, IC sup < 0.5 en todo m): "
          f"{'CUMPLIDO' if (t.ic95_sup < .5).all() else 'NO cumplido: revisar el umbral kNN en el toro'}")

# ---- T4: orden espectral frente a aleatorio en m = M ----
if A is not None and B is not None:
    res = P.order_test(A, B, M)
    display(res.style.format({'fisher_p': '{:.3g}', 'mannwhitney_p': '{:.3g}', 'mse_espectral': '{:.4g}',
                              'mse_aleatorio': '{:.4g}'}).set_caption(f'T4 · m = {M}: mismos modos, misma clase, distinto camino'))
    for _, r in res.iterrows():
        tipo = 'control (deberían ser iguales)' if r.model == 'vae' else 'prueba'
        print(f"{r.model:9s} {tipo}: Fisher p = {r.fisher_p:.3g}")
    display(pd.DataFrame({'espectral': P.information_table(A)['mediana'],
                          'aleatorio': P.information_table(B)['mediana']}).style.format('{:.3g}')
            .set_caption('Información de cada prefijo (oráculo afín, mediana); en m = M son los mismos modos'))

# ---- error angular de la reconstrucción en m = M ----
def mae(d, model, m):
    q = P.with_chain_start(d)
    q = q[(q.model == model) & (q.m == m)]
    v = np.array([ast.literal_eval(s) if isinstance(s, str) else s for s in q.angular_mae_deg])
    return np.median(v, 0) if len(v) else None
for tag, d in (('A', A), ('C', C)):
    if d is not None:
        for model in ('vae', 'vae_warm'):
            v = mae(d, model, M)
            if v is not None:
                print(f'{tag} {model:9s} m={M}: error angular mediano (θ1, θ2) = {v[0]:.2f}°, {v[1]:.2f}°')

In [ ]:
# ---- Figura resumen (2x2) ----
fig, ax = plt.subplots(2, 2, figsize=(12.5, 8.6))
(a, b), (c, e) = ax
if A is not None:
    o = A[A.model == 'oracle'].groupby('m').chart_mse_z
    a.plot(o.median().index, o.median().values, 'o-', color='C4', label='carta: oráculo afín (mediana)')
    a.fill_between(o.median().index, o.quantile(.25).values, o.quantile(.75).values, color='C4', alpha=.2, lw=0)
    x_ = A[A.model == 'affine_x'].groupby('m').mse_x.median()
    a.plot(x_.index, x_.values, 's--', color='gray', label='ambiente: lectura afín de x (mediana)')
    for m, v in PRED_Z.items():
        a.scatter([m], [v], marker='*', s=170, facecolor='none', edgecolor='C4', lw=1.3, zorder=5)
    for m, v in PRED_X.items():
        a.scatter([m], [v], marker='*', s=170, facecolor='none', edgecolor='gray', lw=1.3, zorder=5)
    a.scatter([], [], marker='*', s=170, facecolor='none', edgecolor='k', label='predicción del teorema')
a.axvspan(3, 4, color='C4', alpha=.1, label='posiciones del par de θ₂')
a.set(xscale='log', yscale='log', xlabel='m', ylabel='MSE', title='(a) Información del prefijo')
a.legend(fontsize=7, loc='lower left')

estilos = [('A_k4_spectral', 'vae_warm', 'C4', '-', 'o', 'latente 4, continuación'),
           ('A_k4_spectral', 'vae', 'C0', ':', '^', 'latente 4, desde cero'),
           ('B_k4_shuffled', 'vae_warm', 'C1', '--', 's', 'latente 4, continuación, orden aleatorio'),
           ('C_k2_spectral', 'vae_warm', 'C3', '-', 'D', 'latente 2, continuación'),
           ('C_k2_spectral', 'vae', 'C7', ':', 'v', 'latente 2, desde cero')]
for i, (tag, model, col, ls, mk, lab) in enumerate(estilos):
    if tag in tablas:
        P.errorbar_rates(b, tablas[tag], model, dx=1.03**(i-2), color=col, ls=ls, marker=mk, label=lab)
for tag, col, lab in (('D_plainvae_k4', 'C2', 'plainvae latente 4'), ('D_plainvae_k2', 'C8', 'plainvae latente 2')):
    if tag in tablas and len(tablas[tag]):
        b.axhline(tablas[tag].tasa.iloc[0], color=col, ls='-.', lw=1, label=f'{lab} ({tablas[tag].desplegadas.iloc[0]})')
b.axvspan(3, 4, color='C4', alpha=.1)
b.set(xscale='log', ylim=(-.05, 1.05), xlabel='m', ylabel='fracción con carta desplegada',
      title='(b) Recuperación de la carta (IC 95 %)')
b.legend(fontsize=7)

rng = np.random.default_rng(0)
for j, (tag, model, col, lab) in enumerate((('A_k4_spectral', 'vae_warm', 'C4', 'latente 4, continuación'),
                                            ('A_k4_spectral', 'vae', 'C0', 'latente 4, desde cero'),
                                            ('C_k2_spectral', 'vae_warm', 'C3', 'latente 2, continuación'))):
    if R[tag] is None:
        continue
    g = P.with_chain_start(R[tag])
    g = g[(g.model == model) & g.knn_chart_ratio.notna()]
    c.scatter(g.m*np.exp(rng.uniform(-.06, .06, len(g)) + (j-1)*.08), g.knn_chart_ratio, s=12, alpha=.6,
              color=col, label=lab)
c.axhline(float(COMMON['unfold_ratio']), color='k', ls=':', lw=.8, label='umbral (10× piso)')
c.set(xscale='log', yscale='log', xlabel='m', ylabel='kNN carta / piso',
      title='(c) Distribución de la recuperación (cada punto = réplica)')
c.legend(fontsize=7)

for tag, model, col, ls, mk, lab in estilos:
    if R[tag] is not None:
        q = P.with_chain_start(R[tag])
        q = q[q.model == model].groupby('m').mse_x.median()
        e.plot(q.index, q.values, color=col, ls=ls, marker=mk, label=lab)
for tag, col, lab in (('D_plainvae_k4', 'C2', 'plainvae latente 4'), ('D_plainvae_k2', 'C8', 'plainvae latente 2')):
    if R[tag] is not None:
        e.axhline(P.trained(R[tag]).mse_x.median(), color=col, ls='-.', lw=1, label=lab)
if A is not None and 'chart_decoder' in set(A.model):
    e.axhline(A[A.model == 'chart_decoder'].mse_x.median(), color='k', ls=':', lw=.8, label='decoder sobre la carta verdadera')
e.axhline(.09, color='gray', ls='--', lw=.8, label='piso ambiental sin θ₂ (0.09)')
e.set(xscale='log', yscale='log', xlabel='m', ylabel='test MSE$_x$ (mediana)', title='(d) Reconstrucción')
e.legend(fontsize=7)
for axis in ax.flat:
    axis.set_xticks(LADDER); axis.set_xticklabels([str(m) for m in LADDER], fontsize=8); axis.minorticks_off()
fig.tight_layout()
fig.savefig('/kaggle/working/torus_protocolo.png', dpi=200); fig.savefig('/kaggle/working/torus_protocolo.pdf')
plt.show()
if tablas:
    pd.concat(tablas.values()).to_csv('/kaggle/working/torus_protocolo_tablas.csv', index=False)
# Ramachandran-like panels of angles (reference / oracle / vae) produced by plots.py:
for tag in ('A_k4_spectral', 'C_k2_spectral'):
    for name in ('angles_oracle_reconstructed', 'angles_vae_reconstructed', 'angles_generated_vae_warm'):
        f = Path(f'/kaggle/working/results_torus_{tag}') / f'{name}.png'
        if f.exists():
            print(tag, f.name); display(Image(filename=str(f)))

## Qué descargar

`/kaggle/working/results_torus_*`, `beta_sweep_torus/`, `torus_protocolo.png/.pdf` y `torus_protocolo_tablas.csv`. El `provenance.json` de cada semilla guarda β, la permutación de modos (B) y los controles espectrales.